# pivotai — Evaluation Results Analysis

Visualizes Phase 4 evaluation results across all models.
Charts are saved to `data/evals/charts/` and embedded in `README.md`.

Run from the project root:
```bash
jupyter notebook phase4_evals/notebooks/results_analysis.ipynb
# or execute non-interactively:
# jupyter nbconvert --to notebook --execute phase4_evals/notebooks/results_analysis.ipynb
```

In [ ]:
import json, sys, glob
from pathlib import Path

# Ensure project root is importable
ROOT = Path('').resolve()
while not (ROOT / 'config.py').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from config import EVALS_DIR

CHARTS_DIR = EVALS_DIR / 'charts'
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

# Load latest summary
summaries = sorted(EVALS_DIR.glob('summary_*.json'))
assert summaries, 'No summary_*.json found — run: python phase4_evals/compare.py'
with open(summaries[-1]) as f:
    summary = json.load(f)

print(f'Summary: {summaries[-1].name}')
print(f'Models:  {list(summary["models"].keys())}')

In [ ]:
import matplotlib
matplotlib.use('Agg')  # headless rendering
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

plt.rcParams.update({
    'figure.facecolor': '#1a1a2e',
    'axes.facecolor':   '#16213e',
    'axes.edgecolor':   '#e0e0e0',
    'text.color':       '#e0e0e0',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#e0e0e0',
    'ytick.color':      '#e0e0e0',
    'grid.color':       '#2a2a4a',
    'font.family':      'DejaVu Sans',
})

MODEL_COLORS = {
    'pivotai-ft':         '#4fc3f7',
    'pivotai-distill':    '#81c784',
    'pivotai-curriculum': '#ffb74d',
    'llama3.1:8b':         '#ef5350',
}
MODEL_SHORT = {
    'pivotai-ft':         'ft',
    'pivotai-distill':    'distill',
    'pivotai-curriculum': 'curriculum',
    'llama3.1:8b':         'baseline',
}
TARGETS = summary['targets']
MODELS  = [m for m in MODEL_COLORS if m in summary['models']]
print('Models to plot:', MODELS)

In [ ]:
# ── Chart 1: Radar chart — all metrics × all models ──────────────────────────

RADAR_METRICS = [
    'json_valid', 'savings_valid', 'budget_compliance', 'schema_compliance',
    'intent_alignment', 'rouge_l', 'bertscore_f1',
    'reasoning_coherence', 'grounding_accuracy',
]
RADAR_LABELS = [
    'JSON valid', 'Savings found', 'Budget comply', 'Schema comply',
    'Intent align', 'ROUGE-L', 'BERTScore F1',
    'Coherence', 'Grounding',
]

N = len(RADAR_METRICS)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]  # close polygon

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.set_facecolor('#16213e')
fig.patch.set_facecolor('#1a1a2e')

for model in MODELS:
    mdata = summary['models'][model]
    values = [mdata.get(m) or 0.0 for m in RADAR_METRICS]
    values += values[:1]
    color = MODEL_COLORS[model]
    ax.plot(angles, values, 'o-', linewidth=2, color=color, label=MODEL_SHORT[model])
    ax.fill(angles, values, alpha=0.15, color=color)

# Target circle
target_vals = [TARGETS.get(m, 0.0) for m in RADAR_METRICS] + [TARGETS.get(RADAR_METRICS[0], 0.0)]
ax.plot(angles, target_vals, '--', linewidth=1, color='#ffffff', alpha=0.4, label='Target')

ax.set_xticks(angles[:-1])
ax.set_xticklabels(RADAR_LABELS, size=10)
ax.set_ylim(0, 1.05)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25', '0.50', '0.75', '1.0'], size=8)
ax.grid(color='#2a2a4a', linewidth=0.5)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), framealpha=0.3, fontsize=10)
ax.set_title('pivotai Model Comparison — All Metrics', pad=20, size=13, fontweight='bold')

out = CHARTS_DIR / 'radar_all_metrics.png'
fig.savefig(out, dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.close()
print(f'Saved: {out.name}')

In [ ]:
# ── Chart 2: Head-to-head win rates ───────────────────────────────────────────

h2h = summary.get('head_to_head', {})
pairs, win_a, win_b = [], [], []

for key, vals in h2h.items():
    a, b = key.split('_vs_')
    short_a = MODEL_SHORT.get(a, a.replace('pivotai-', ''))
    short_b = MODEL_SHORT.get(b, b.replace('pivotai-', ''))
    total   = vals['wins_a'] + vals['wins_b'] + vals.get('ties', 0) or 1
    pairs.append(f'{short_a}\nvs\n{short_b}')
    win_a.append(vals['wins_a'] / total * 100)
    win_b.append(vals['wins_b'] / total * 100)

x = np.arange(len(pairs))
w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x - w/2, win_a, w, color='#4fc3f7', label='Model A wins')
ax.bar(x + w/2, win_b, w, color='#ef9a9a', label='Model B wins')
ax.axhline(50, color='white', linewidth=0.8, linestyle='--', alpha=0.4, label='50% (coin flip)')

for i, (a, b) in enumerate(zip(win_a, win_b)):
    ax.text(i - w/2, a + 1, f'{a:.0f}%', ha='center', va='bottom', size=9)
    ax.text(i + w/2, b + 1, f'{b:.0f}%', ha='center', va='bottom', size=9)

ax.set_xticks(x)
ax.set_xticklabels(pairs, fontsize=9)
ax.set_ylabel('Win rate (%)')
ax.set_ylim(0, 100)
ax.set_title('Head-to-Head Win Rates (LLM Judge, 92 cases)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

out = CHARTS_DIR / 'head_to_head.png'
fig.savefig(out, dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.close()
print(f'Saved: {out.name}')

In [ ]:
# ── Chart 3: Structural vs semantic metrics grouped bar ───────────────────────

STRUCTURAL = ['json_valid', 'savings_valid', 'budget_compliance', 'schema_compliance']
SEMANTIC   = ['intent_alignment', 'rouge_l', 'bertscore_f1', 'reasoning_coherence', 'grounding_accuracy']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Structural vs Semantic Metrics', fontsize=13, fontweight='bold')

for ax, metric_group, title in [
    (axes[0], STRUCTURAL, 'Structural Metrics'),
    (axes[1], SEMANTIC,   'Semantic Metrics'),
]:
    n_metrics = len(metric_group)
    n_models  = len(MODELS)
    x = np.arange(n_metrics)
    bar_w = 0.7 / n_models

    for i, model in enumerate(MODELS):
        mdata  = summary['models'][model]
        vals   = [mdata.get(m) or 0.0 for m in metric_group]
        offset = (i - n_models / 2 + 0.5) * bar_w
        bars   = ax.bar(x + offset, vals, bar_w,
                        label=MODEL_SHORT[model], color=MODEL_COLORS[model], alpha=0.85)

    # Target line per metric
    for j, m in enumerate(metric_group):
        t = TARGETS.get(m, 0)
        ax.hlines(t, j - 0.45, j + 0.45, colors='white', linestyles='--', linewidth=1, alpha=0.5)

    short_labels = [m.replace('_', '\n') for m in metric_group]
    ax.set_xticks(x)
    ax.set_xticklabels(short_labels, fontsize=8)
    ax.set_ylim(0, 1.1)
    ax.set_title(title, fontsize=11)
    ax.legend(fontsize=8, loc='lower right')
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylabel('Score')

out = CHARTS_DIR / 'structural_vs_semantic.png'
fig.savefig(out, dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.close()
print(f'Saved: {out.name}')

In [ ]:
# ── Chart 4: Red-team pass rates ──────────────────────────────────────────────

names  = [MODEL_SHORT[m] for m in MODELS]
passes = [summary['models'][m].get('red_team_pass') or 0.0 for m in MODELS]
colors = [MODEL_COLORS[m] for m in MODELS]
target = TARGETS['red_team_pass']

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(names, passes, color=colors, alpha=0.85, edgecolor='white', linewidth=0.5)
ax.axhline(target, color='#ff6b6b', linewidth=2, linestyle='--', label=f'Target ({target:.0%})')

for bar, val in zip(bars, passes):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.01,
            f'{val:.1%}', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylim(0, 1.05)
ax.set_ylabel('Pass rate')
ax.set_title('Red-Team Pass Rates (45 adversarial cases, 15 per model)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

out = CHARTS_DIR / 'red_team_pass.png'
fig.savefig(out, dpi=150, bbox_inches='tight', facecolor='#1a1a2e')
plt.close()
print(f'Saved: {out.name}')
print(f'\nAll charts saved to: {CHARTS_DIR}')